# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/FLYRANK-INTERN/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. Watch three numbers, then a hand-written rule that reads its own
work: **(1)** two signal checks against the warehouse, **(2)** one transparent rule scored into a ranked queue
saved to `work/outputs/baseline_action_score.csv`, **(3)** a top-10 read with a skeptic's eye, **(4)** weak picks
and a leakage check, **(5)** self-check. All numerals are measured on the warehouse, March 2026 decision slice —
the same slice and labels the Week-5 model will be held to.

## 1. My rule, in plain words first

> **Review a page first when it still has real search demand this month but is under-clicking for the position it
> holds.** Two numbers do all the work: how far below its position-neighbours its click-through sits (**risk**),
> and how many March impressions are at stake (**impact**). Pages too small to act on, or too deep / unpositioned
> to have a CTR story, stay in the monitor queue.

Reason codes the queue can output (one per row):

| reason_code | meaning | action |
|---|---|---|
| `low_ctr_high_position` | page-one page (top 10) in the bottom third of click-through for its position band | `refresh_and_review_ctr` |
| `ctr_shortfall` | positioned page that still under-clicks its band, or page-2/deep under-clicking | `refresh` |
| `review_baseline` | clears the demand floor but no strong CTR story | `monitor` |

In [1]:
import os, json, duckdb, pandas as pd, numpy as np
from pathlib import Path

# anchor to the repo root no matter where the notebook is executed from
root = Path(".").resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
OUT = root / "work" / "outputs"

def hf_token():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        import getpass
        tok = getpass.getpass("HF_TOKEN: ")
    return tok

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + hf_token() + "')")
con.execute("SET http_timeout = 900")
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DEC, LAB = "2026-03", "2026-04"

ff = con.sql(f'''
    WITH dec AS (
      SELECT * FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={DEC}/data_0.parquet')
      WHERE gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions)                                    AS imp_march,
           SUM(gsc_clicks)                                         AS clk_march,
           AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_march
    FROM dec GROUP BY 1, 2
''').df()

lab = con.sql(f'''
    SELECT content_hash_id, SUM(gsc_impressions) AS apr_imp
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={LAB}/data_0.parquet')
    WHERE gsc_data_available IS TRUE GROUP BY 1
''').df()

lane = ff.merge(lab, on="content_hash_id", how="inner")
lane = lane[lane["imp_march"] >= 100].reset_index(drop=True)   # demand floor, matches w03 slice
lane["declined_next_30d"] = (lane["apr_imp"] < 0.8 * lane["imp_march"]).astype(int)
lane["ctr"] = lane["clk_march"] / lane["imp_march"]

print("decision-month rows  :", len(ff))
print("lane pool (imp>=100) :", len(lane), " rows")
print("decline base rate    :", round(lane["declined_next_30d"].mean(), 3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

decision-month rows  : 176738
lane pool (imp>=100) : 100893  rows
decline base rate    : 0.515


### Check 1 — Volume (the quick-win flag's signal)

**Is higher March demand a reason to act?** Bucket the pool by impression tier; the flag cares because a flag is
only worth setting when there is traffic on the line. A verdict explains *this* data.

In [2]:
tiers = pd.cut(lane["imp_march"], bins=[0, 1_000, 10_000, np.inf],
               labels=["low (100-1k)", "moderate (1k-10k)", "high (10k+)"])
sigA = lane.groupby(tiers, observed=True).agg(
    n=("content_hash_id", "count"),
    decline_rate=("declined_next_30d", "mean"),
    median_ctr=("ctr", "median"),
).round(3)
sigA["pool_share"] = (sigA["n"] / sigA["n"].sum() * 100).round(1)
print("SIGNAL CHECK 1 - volume (quick-win signal), n printed per bucket:")
print(sigA.to_string())
print()
print("VERDICT: MIXED - volume gates the pool (n), but it is not a risk driver:")
print("high-demand pages decline LESS (0.45) than low-demand ones (0.54).")
print("Survives the check as a GATE, not a score driver - the rule uses it for impact, not risk.")


SIGNAL CHECK 1 - volume (quick-win signal), n printed per bucket:
                       n  decline_rate  median_ctr  pool_share
imp_march                                                     
low (100-1k)       55890         0.538       0.000        55.4
moderate (1k-10k)  39126         0.492       0.002        38.8
high (10k+)         5877         0.445       0.002         5.8

VERDICT: MIXED - volume gates the pool (n), but it is not a risk driver:
high-demand pages decline LESS (0.45) than low-demand ones (0.54).
Survives the check as a GATE, not a score driver - the rule uses it for impact, not risk.


### Check 2 — CTR vs position (the CTR-fix logic's signal)

**Do pages that sit high in results but under-click actually decline?** Backbone of the session's CTR-fix logic.
Take visible, positioned pages (imp ≥ 500, valid position), split each position band into click-through thirds,
and watch the decline rate across the grid.

In [3]:
vis = lane[(lane["pos_march"] > 0) & (lane["imp_march"] >= 500)].copy()
def pos_band(p):
    if pd.isna(p) or p <= 0: return "unknown"
    if p <= 3:  return "top3"
    if p <= 10: return "p1"
    if p <= 20: return "p2"
    return "deep"
vis["band"] = vis["pos_march"].map(pos_band)

vis["ctr_tercile"] = pd.Series(pd.NA, index=vis.index, dtype="object")
for b in ["top3", "p1", "p2", "deep"]:
    m = vis["band"].eq(b)
    if m.sum() < 3: continue
    ranks = vis.loc[m, "ctr"].rank(method="first")
    vis.loc[m, "ctr_tercile"] = pd.qcut(ranks, 3, labels=["bottom", "mid", "top"]).astype("object").values

sigB = vis.groupby(["band", "ctr_tercile"], observed=True).agg(
    n=("content_hash_id", "count"),
    decline_rate=("declined_next_30d", "mean"),
    median_ctr=("ctr", "median"),
).round(3)
print("SIGNAL CHECK 2 - CTR vs position (CTR-fix signal), n printed per bucket:")
print(sigB.to_string())
print()
print("VERDICT: CONFIRMED - page-one pages (top3/p1) in the bottom CTR third decline at 0.62-0.71,")
print("while the top third of the same band sits at 0.30. Two-to-one, monotone, and the edge")
print("fades as position worsens. This is the signal my rule leans on.")


SIGNAL CHECK 2 - CTR vs position (CTR-fix signal), n printed per bucket:
                      n  decline_rate  median_ctr
band ctr_tercile                                 
deep bottom        3731         0.577       0.000
     mid           3731         0.524       0.001
     top           3731         0.469       0.003
p1   bottom       10609         0.621       0.001
     mid          10609         0.501       0.002
     top          10609         0.309       0.005
p2   bottom        3941         0.621       0.000
     mid           3941         0.548       0.002
     top           3942         0.394       0.005
top3 bottom        2334         0.711       0.001
     mid           2334         0.581       0.003
     top           2334         0.302       0.007

VERDICT: CONFIRMED - page-one pages (top3/p1) in the bottom CTR third decline at 0.62-0.71,
while the top third of the same band sits at 0.30. Two-to-one, monotone, and the edge
fades as position worsens. This is the signal my

## 2. Build the ranked queue

Rule encoding (readable, hand-set from the checked buckets — no fitted weights):

```python
score = risk_tier(band, ctr_tercile) * log1p(imp_march)   # risk x impact
```

| risk_tier | condition (from Check 2) |
|---|---|
| 3.0 | top3 band, bottom CTR third |
| 2.5 | p1 band, bottom CTR third |
| 2.0 | top3 band, middle CTR third |
| 1.5 | p1 band, middle CTR third |
| 1.0 | any other positioned, visible page |
| 0.0 | below the visibility floor, or no valid position → `monitor` |

The queue is written to `work/outputs/baseline_action_score.csv` — deliberately kept out of git; the notebook
regenerates it on every run. The label is used for checking ONLY in the next cell; it never enters the rule.

In [4]:
df = lane.copy()
df["band"] = df["pos_march"].map(pos_band)
df["ctr_tercile"] = pd.Series(pd.NA, index=df.index, dtype="object")
for b in ["top3", "p1", "p2", "deep"]:
    m = df["band"].eq(b) & (df["imp_march"] >= 500)
    if m.sum() < 3: continue
    ranks = df.loc[m, "ctr"].rank(method="first")
    df.loc[m, "ctr_tercile"] = pd.qcut(ranks, 3, labels=["bottom", "mid", "top"]).astype("object").values

df["risk"] = 0.0
df.loc[df["band"].eq("top3") & df["ctr_tercile"].eq("bottom"), "risk"] = 3.0
df.loc[df["band"].eq("p1")   & df["ctr_tercile"].eq("bottom"), "risk"] = 2.5
df.loc[df["band"].eq("top3") & df["ctr_tercile"].eq("mid"),    "risk"] = 2.0
df.loc[df["band"].eq("p1")   & df["ctr_tercile"].eq("mid"),    "risk"] = 1.5
df.loc[df["risk"].eq(0.0) & df["band"].isin(["top3", "p1", "p2", "deep"]) & (df["imp_march"] >= 500), "risk"] = 1.0
df["score"] = df["risk"] * np.log1p(df["imp_march"])

df["reason_code"] = "review_baseline"
df.loc[(df["band"].isin(["top3", "p1"])) & df["ctr_tercile"].eq("bottom"), "reason_code"] = "low_ctr_high_position"
df.loc[df["reason_code"].eq("review_baseline") & (df["score"] > 0), "reason_code"] = "ctr_shortfall"
df["action"] = np.where(df["reason_code"].eq("low_ctr_high_position"), "refresh_and_review_ctr",
                  np.where(df["reason_code"].eq("ctr_shortfall"), "refresh", "monitor"))

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

# the queue is a product artifact: the label and the April window never ride along
queue = queue.drop(columns=["declined_next_30d", "apr_imp"])

csv_cols = ["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action",
            "imp_march", "clk_march", "pos_march", "active_days_march", "ctr", "band", "ctr_tercile"]
os.makedirs(OUT, exist_ok=True)
queue[csv_cols].to_csv(OUT / "baseline_action_score.csv", index=False)

print("queue written:", OUT / "baseline_action_score.csv")
print("rows:", len(queue), "| score range:", round(queue["score"].min(), 1), "-", round(queue["score"].max(), 1))
print("actions:", queue["action"].value_counts().to_dict())


queue written: C:\Users\Dell\OneDrive\Desktop\opencode\FLYRANK-INTERN\work\outputs\baseline_action_score.csv
rows: 100893 | score range: 0.0 - 36.7
actions: {'refresh': 48903, 'monitor': 39047, 'refresh_and_review_ctr': 12943}


In [5]:
# honest check against the SAME label the model will be held to (April drop, w03 definition)
label_map = lane.set_index("content_hash_id")["declined_next_30d"]
check = queue.assign(y=queue["content_hash_id"].map(label_map))

pk = {k: round(check.head(k)["y"].mean(), 3) for k in (10, 20, 50, 100)}
base = round(lane["declined_next_30d"].mean(), 3)
print("base rate:", base, "(random pick at K ~= this, so a baseline must clear it)")
print("precision@10:", pk[10], "| @20:", pk[20], "| @50:", pk[50], "| @100:", pk[100])

metrics = {"task": "ml-07", "decision_month": DEC, "label_month": LAB,
           "queue_rows": int(len(queue)), "base_rate": base, "precision_at_k": pk,
           "action_counts": queue["action"].value_counts().to_dict()}
json.dump(metrics, open(OUT / "w04_baseline_metrics.json", "w"), indent=2)
print("metrics receipt written:", OUT / "w04_baseline_metrics.json")


base rate: 0.515 (random pick at K ~= this, so a baseline must clear it)
precision@10: 0.7 | @20: 0.7 | @50: 0.74 | @100: 0.69
metrics receipt written: C:\Users\Dell\OneDrive\Desktop\opencode\FLYRANK-INTERN\work\outputs\w04_baseline_metrics.json


## 3. Top-10 review

Each of the top ten, one line: the action, why it is there, and what would make it wrong. The lines are
generated from live queue rows, not typed in after the fact.

Each line below is generated from a live top-10 queue row:
**action — why it is there — what would make it wrong**.

In [6]:
def why_line(r):
    pos = "pos ~" + "{:.1f}".format(r["pos_march"]) + " (" + r["band"] + ")"
    ctr = "CTR " + "{:.2%}".format(r["ctr"]) + " = " + r["ctr_tercile"] + " third for that band"
    imp = "~{:,.0f} March imps at stake".format(r["imp_march"])
    return pos + " + " + ctr + " + " + imp

def wrong_line(r):
    if r["clk_march"] == 0:
        return "zero clicks may be a tracking/title artifact, not real demand loss"
    if r["active_days_march"] < 15:
        return "position/CTR averaged over too few active days this month"
    return "position is a 31-day average - a late-March slide is hidden; or CTR recovers on its own"

for _, r in queue.head(10).iterrows():
    print("{:>4} | ".format(int(r["rank"]))
          + "{:<26}".format(r["action"]) + " | " + why_line(r) + " | wrong if " + wrong_line(r))


   1 | refresh_and_review_ctr     | pos ~2.6 (top3) + CTR 0.14% = bottom third for that band + ~203,497 March imps at stake | wrong if position is a 31-day average - a late-March slide is hidden; or CTR recovers on its own
   2 | refresh_and_review_ctr     | pos ~2.5 (top3) + CTR 0.10% = bottom third for that band + ~89,229 March imps at stake | wrong if position is a 31-day average - a late-March slide is hidden; or CTR recovers on its own
   3 | refresh_and_review_ctr     | pos ~1.5 (top3) + CTR 0.04% = bottom third for that band + ~80,821 March imps at stake | wrong if position is a 31-day average - a late-March slide is hidden; or CTR recovers on its own
   4 | refresh_and_review_ctr     | pos ~2.6 (top3) + CTR 0.16% = bottom third for that band + ~73,272 March imps at stake | wrong if position is a 31-day average - a late-March slide is hidden; or CTR recovers on its own
   5 | refresh_and_review_ctr     | pos ~1.6 (top3) + CTR 0.06% = bottom third for that band + ~70,398 March im

## 4. Weak picks + leakage check

Every hand-written rule has its own bad ideas. Name the patterns honestly, then confirm the queue is clean:
no label and no future-window feature made it into the CSV.

In [7]:
top_n = queue.head(200)
zero_click = top_n["clk_march"].eq(0)
few_days = top_n["active_days_march"].lt(15)

print("Weak patterns in the top-200 of the queue:")
print("  - zero-click rows          : n =", zero_click.sum(),
      "(ranks", list(top_n[zero_click]["rank"][:6]), ")")
print("  - averaged on <15 active days : n =", few_days.sum())
print("  - deep-position rows (p2/deep): n =", int(top_n["band"].isin(["p2", "deep"]).sum()))
print()
print("Why these are weak: a CTR story needs real clicks and real coverage - rank 185 is a zero-click page")
print("flying the CTR flag (likely a tracking artifact), and the <15-active-day page averages its position/CTR")
print("over a thin month, so one bad week can fake a bottom-third grade. No p2/deep rows reached the top-200.")
print()

# leakage guard: the CSV must carry only March inputs + the scored outputs, never the label or April
assert "declined_next_30d" not in queue.columns, "label leaked into queue"
assert "apr_imp" not in queue.columns, "future-window value leaked into queue"
print("Queue columns (March inputs + score/reason/action only):")
print(sorted(csv_cols))
print()
print("Care words in this notebook: MIXED / CONFIRMED are verdicts on *observed* buckets;")
print("P@K is *measured* on the April label; the rule is decision-support, not a forecast.")


Weak patterns in the top-200 of the queue:
  - zero-click rows          : n = 1 (ranks [185] )
  - averaged on <15 active days : n = 1
  - deep-position rows (p2/deep): n = 0

Why these are weak: a CTR story needs real clicks and real coverage - rank 185 is a zero-click page
flying the CTR flag (likely a tracking artifact), and the <15-active-day page averages its position/CTR
over a thin month, so one bad week can fake a bottom-third grade. No p2/deep rows reached the top-200.

Queue columns (March inputs + score/reason/action only):
['action', 'active_days_march', 'band', 'client_hash_id', 'clk_march', 'content_hash_id', 'ctr', 'ctr_tercile', 'imp_march', 'pos_march', 'rank', 'reason_code', 'score']

Care words in this notebook: MIXED / CONFIRMED are verdicts on *observed* buckets;
P@K is *measured* on the April label; the rule is decision-support, not a forecast.


### Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed locally with `HF_TOKEN`)
- [x] No client names, URLs, or private queries anywhere — only hashes and aggregates
- [x] Claims use careful words: observed, measured, verdicts, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit repo URL on the card. Done.